# Prior rigs — drones drawn from the generic prior

Ten drones from `experiments.noise_model.drone_prior.sample_prior` (laws grounded in the static-rig campaign: blade-pass comb written as line-over-floor — $\mathcal N(22,7)$ dB per rotor at $k=2$, a power-law decay down to a $\mathcal N(-4,4)$ dB tail that rides the floor to the last order (no cliff; Michael's fits score at $\chi^2_7$ tails 0.18 / 0.05 under these laws), converted to `profile_db` through the model's own expected periodogram, odd-order penalty growing with $\log k$, motor commutation family at multiples of $3p$, per-rotor scatter, bench shaft OU $(\sigma_\nu, \lambda)$, one per-order log-OU amplitude envelope shared by the mics that absorbs AM and the fast phase wobble, per-(mic, line) scatter redrawn per clip, floor spline about a hump/tilt template, SC wind on some capsules from none up to the DREGON free-flight level), each **gated** on trajectory-hyperprior windows (identifiability rule with the amplitude pedestal in the line shapes: 3 dB, ≥ 2 orders per rotor in 80 % of its eligible frames; the 6 dB count is reported for information).

Each drone is rendered into three 8-channel, 8 s recordings on fresh hyperprior trajectories (`prior_rigs_lab.build`; cached under `results/prior_rigs/<name>/` and `.cache/prior_rigs/<name>/`). The rig view is `noise_lab.param_view` (as `plot_v2_params` in `noise_lab.ipynb`): stems = expected line peaks at 80 rev/s, grey = floor, red caps = shaft width — the per-order AM pedestals are not part of that expectation. The recordings are examined with `plots.spectrum_viewer`.


In [ ]:
%matplotlib inline
import prior_rigs_lab as L

lab = L.build("prior_v1_seed0", seed=0, n_rigs=10, n_rec=3, duration_s=8.0)  # cached under results/ and .cache/
print({k: v for k, v in lab.stats.items() if k != "reasons"})


## The ten rigs

The parameter view of `noise_lab` (`rig_view_plotly`): pick a rig, then inside the figure `all rotors` / `rotor mean` / one rotor (stems at $k f_r$, red caps = $\pm$ half width) and the common rotor speed on the slider. Grey = the floor with the comb off. The AM pedestals are not in this expectation.


In [ ]:
L.rig_browser(lab)  # rig dropdown; inside: all rotors / rotor mean / one rotor, speed slider


## Recordings

Pick a rig and a recording: the trajectory (hyperprior drone, airborne process) and the spectrum viewer (channel dropdown, spectrogram / spectrum, zoom).

In [ ]:
L.browser(lab)


## A rig on a real clip's trajectory

Pick a clip of the frozen real validation split (`DREGON-LM-V4-michaels-valid-full`: 0–13 DREGON with speech, 14–21 DREGON without, 22–36 MD2) and a rig: the rig is rendered on the clip's own rotor track and scaled to the clip's RMS; the viewer overlays the two spectra (channel dropdown, spectrogram toggle), with a player for each.


In [ ]:
L.compare_browser(lab)


Or directly:

In [ ]:
L.plot_trajectory(lab, 0, 0)
L.view(lab, 0, 0)


## The bank rigs before and after the phase pedestal (proposal, 2026-10-25)

The prior arms trained on the 16 384 rigs of `prior-v1-banks`. Their conversion to noise model v4 (`experiments.noise_model.prior_v4.convert_prior_v1`, run here under the current rules `prior_v4/3`; `docs/experiments/rp-synthetic.md` § "The prior_v1 rigs in noise model v4") keeps each rig's comb, floor, speed laws, shaft and pedestal and makes the motor family a coherent motor line. The rigs carry no wind: as in v1, it is drawn per clip at render time (`render_noise(..., wind_sc=...)`, v1's law, 2026-10-25). v1's pedestal is split at the bank's amplitude cap (`prior_v4.AMP_CAPS`, raised 2026-10-25 to the larger of the fits' cap and the A0 bench p90: 0.075 / 0.61 / 0.24 / 0.21 / 0.23 for $k$ 1–3 / 4–8 / 9–20 / 21–40 / 41+); the rest becomes a phase part at the same slow rate. The lines stay needles: v1 measured only the slow amplitude flicker (0.25 s frames) and gave the phase to one quiet shaft.

The proposal adds a **phase pedestal** per order (`experiments.noise_model.prior_v4.with_phase_pedestal`):

- $\sigma^2_\phi$ and the mean half-width $W$ are log-normal around the bench's direct measurement per order group (step A0, 20 DREGON recordings, blade-pass orders; black lines in the first figure): $\sigma^2_\phi$ 0.04 / 0.20 / 0.96 / 1.70 / 1.67 and $W$ 0.8 / 0.8 / 1.6 / 4.9 / 8.4 Hz for $k$ 1–3 / 4–8 / 9–20 / 21–40 / 41+;
- caps $\sigma^2_\phi \le 3$ (the carrier keeps ≥ 5 % of the line) and $W \le 24$ Hz (half the slowest bench line spacing);
- no floor guard in this comparison (`prior_rigs_lab.PED_GUARD = False`, 2026-10-25). The guard (`prior_v4.guard_phase_pedestal`: skirts 3 dB under the wind-free floor halfway between lines at 80 rev/s, $\sigma^2_\phi$ shrunk until they are) cut the total pedestal below the converted one on most of the orders it touched; set `PED_GUARD = True` to compare.

Each rig is rendered three ways on **one** trajectory with one seed: the v1 payload as trained, the v4 conversion, the conversion with the phase pedestal. One scale for all three, so loudness differences are real. `steady` holds the four rotors at 72 / 76 / 80 / 84 rev/s, where the line shapes show; `0`–`2` are hyperprior airborne windows (the line-shape spectra then use the steadiest second, shaded). All three draw the wind per clip from v1's law (probability 0.7); the v1 and v4 renderers consume their streams differently, so the wind draws differ between rows.

Per rig, first the drawn phase pedestal per order, then the **rig profile** (Plotly; speed slider, `all rotors` / one rotor; the three rows share the frequency axis, so a zoom applies to all). All rotors run at the slider's speed; the y axis is dB of power, with densities per 1 Hz, so a needle reads its power in one 1 Hz bin:

- stems: the **needle** of each line (the coherent share $e^{-(\sigma^2_A + \sigma^2_\phi)}$ of its power), blue aerodynamic, orange v4 motor line (no pedestal), purple a v1 aerodynamic line carrying the motor boost (it shares the line's amplitude pedestal); a short thick cap at the top spans the needle's shaft-FM half width (approximate: the smaller of the diffusion Lorentzian and the frozen-rate Gaussian);
- red curve: the summed **pedestal skirts** (amplitude + phase) of all lines; red caps: each line's skirt peak, spanning its mean half width;
- grey: the **floor** (no wind, which every row draws per clip; the v1 row uses the conversion's floor, the same law at equal speeds);
- hover a line's marker for its power, needle share, shaft width and both pedestals (`all rotors`: summed power, power-weighted parameters).


In [ ]:
%matplotlib inline
import prior_rigs_lab as L  # this section needs nothing above it

L.pedestal_browser()  # rig (or any index 0-16383), trajectory, channel


Or directly:


In [ ]:
L.pedestal_compare(13504, "steady", ch=0)
